In [4]:
from pathlib import Path
import json
import difflib

import httpx
from transformers import AutoTokenizer

from mission_control.inference.requests import ModelMessage

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


# Setup


In [5]:
from pathlib import Path
import json
import difflib

import httpx
from transformers import AutoTokenizer

from mission_control.inference.requests import ModelMessage


# ------------------------------------------------------------------
# Local Qwen cache
# ------------------------------------------------------------------

HF_HUB = Path(
    "/Users/lucasleow/Library/CloudStorage/"
    "OneDrive-Personal/LucasTechVault/"
    "Mission Control/"
    "mission-control-local-inference/"
    "hf-cache/hub"
)

MODEL_CACHE = HF_HUB / "models--Qwen--Qwen3.5-0.8B"
SNAPSHOTS_DIR = MODEL_CACHE / "snapshots"

snapshots = [
    path
    for path in SNAPSHOTS_DIR.iterdir()
    if path.is_dir()
]

assert snapshots, f"No snapshots found under {SNAPSHOTS_DIR}"

# Use most recently modified cached snapshot.
MODEL_SNAPSHOT = max(
    snapshots,
    key=lambda path: path.stat().st_mtime,
)

CHAT_TEMPLATE_PATH = MODEL_SNAPSHOT / "chat_template.jinja"

print("Model snapshot:")
print(MODEL_SNAPSHOT)

print("\nChat template:")
print(CHAT_TEMPLATE_PATH)

print("\nTemplate exists:")
print(CHAT_TEMPLATE_PATH.exists())

Model snapshot:
/Users/lucasleow/Library/CloudStorage/OneDrive-Personal/LucasTechVault/Mission Control/mission-control-local-inference/hf-cache/hub/models--Qwen--Qwen3.5-0.8B/snapshots/2fc06364715b967f1860aea9cf38778875588b17

Chat template:
/Users/lucasleow/Library/CloudStorage/OneDrive-Personal/LucasTechVault/Mission Control/mission-control-local-inference/hf-cache/hub/models--Qwen--Qwen3.5-0.8B/snapshots/2fc06364715b967f1860aea9cf38778875588b17/chat_template.jinja

Template exists:
True


## Load Tokenizer Locally


In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_SNAPSHOT),
    local_files_only = True,
)

print("Tokenizer class: ", tokenizer.__class__.__name__)
print("Vocab size: ", len(tokenizer))
print("Special tokens: ", tokenizer.all_special_tokens)

Tokenizer class:  Qwen2Tokenizer
Vocab size:  248077
Special tokens:  ['<|im_end|>', '<|endoftext|>', '<|audio_start|>', '<|audio_end|>', '<|audio_pad|>', '<|image_pad|>', '<|video_pad|>', '<|vision_start|>', '<|vision_end|>']


## Experiment A - Audit Jinja Template


### A1. Print relevant Jinja Lines

Instead of printing entire 8KB template, search for key structural pieces


In [7]:
template_text = CHAT_TEMPLATE_PATH.read_text()

search_terms = [
    "message.role",
    "im_start", "im_end",
    "add_generation_prompt",
    "enable_thinking",
    "<think>", "</think>"
]

print("=" * 80)
print("EXPERIMENT A - RELEVANT JINJA TEMPLATE LINES")
print("=" * 80)

for line_num, line in enumerate(
    template_text.splitlines(),
    start = 1
):
    if any(term in line for term in search_terms):
        print(f"{line_num:03d}: {line}")

EXPERIMENT A - RELEVANT JINJA TEMPLATE LINES
046:     {{- '<|im_start|>system\n' }}
060:     {{- '<|im_end|>\n' }}
064:         {{- '<|im_start|>system\n' + content + '<|im_end|>\n' }}
070:     {%- if ns.multi_step_tool and message.role == "user" %}
083:     {%- if message.role == "system" %}
087:     {%- elif message.role == "user" %}
088:         {{- '<|im_start|>' + message.role + '\n' + content + '<|im_end|>' + '\n' }}
089:     {%- elif message.role == "assistant" %}
094:             {%- if '</think>' in content %}
095:                 {%- set reasoning_content = content.split('</think>')[0].rstrip('\n').split('<think>')[-1].lstrip('\n') %}
096:                 {%- set content = content.split('</think>')[-1].lstrip('\n') %}
101:             {{- '<|im_start|>' + message.role + '\n<think>\n' + reasoning_content + '\n</think>\n\n' + content }}
103:             {{- '<|im_start|>' + message.role + '\n' + content }}
130:         {{- '<|im_end|>\n' }}
131:     {%- elif message.role == "to

## Experiment B - Render Raw String

Pass ModelMessage class through tokenizer

```
Mission Control abstraction
        ↓
Qwen representation
```


### B1. Construct the Messages


In [10]:
model_messages = [
    ModelMessage(
        role="system",
        content=(
            "You are the reasoning engine for Mission Control."
        ),
    ),
    ModelMessage(
        role="user",
        content=(
            "Explain the purpose of ModelGateway "
            "in two sentences."
        ),
    ),
]

messages = [
    message.model_dump() # convert python obj to python dict by Pydantic
    for message in model_messages
]

messages

[{'role': 'system',
  'content': 'You are the reasoning engine for Mission Control.'},
 {'role': 'user',
  'content': 'Explain the purpose of ModelGateway in two sentences.'}]

### B2. Render, but NOT tokenize


In [12]:
rendered_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print("=" * 40)
print("EXPERIMENT B - RAW RENDERED PROMPT")
print("=" * 40)

print(rendered_prompt)


EXPERIMENT B - RAW RENDERED PROMPT
<|im_start|>system
You are the reasoning engine for Mission Control.<|im_end|>
<|im_start|>user
Explain the purpose of ModelGateway in two sentences.<|im_end|>
<|im_start|>assistant
<think>

</think>




In [13]:
print(repr(rendered_prompt))

'<|im_start|>system\nYou are the reasoning engine for Mission Control.<|im_end|>\n<|im_start|>user\nExplain the purpose of ModelGateway in two sentences.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'


## Experiment C - Token Audit

Keep everything similar but set Tokenize = True
